## Environment Setup


In [ ]:
%load_ext watermark


In [ ]:
import itertools as it

from IPython.display import display_html
import numpy as np
import pandas as pd
import pylib  # noqa: F401
from pylib.abm_2026_03_16 import simulate
import seaborn as sns
from teeplot import teeplot as tp
from tqdm.auto import tqdm


In [ ]:
teeplot_subdir = "2026-03-16-abm"


In [ ]:
%watermark -diwmuv -iv


## Plotting Implementation


In [ ]:
def render_timeseries_plots(
    df: pd.DataFrame,
    suptitle: str,
    teeplot_outattrs: dict,
    POP_SIZE: int,
) -> None:
    for what, row in it.product(
        ["Susc", "Strain"],
        ["Seed", None],
    ):
        data = (
            df.filter(regex=f"Step|Seed|{what}", axis=1)
            .melt(
                id_vars=["Step", "Seed"],
                var_name="Class",
                value_name="Prevalence",
            )
            .astype(
                {"Step": int, "Class": str, "Prevalence": float},
            )
        )
        data["Ham. Wt."] = data["Class"].str.count("1|3|5|7|9")
        palette = dict(
            zip(
                data["Class"].unique(),
                sns.color_palette(
                    "colorblind", len(data["Class"].unique())
                ),
            )
        )
        with tp.teed(
            sns.relplot,
            data=data,
            x="Step",
            y="Prevalence",
            hue="Class",
            row=row,
            alpha=0.8,
            dashes=False,
            errorbar=("pi", 100),
            err_kws=dict(alpha=0.1),
            estimator=np.median,
            facet_kws=dict(
                margin_titles=True,
            ),
            kind="line",
            palette=palette,
            teeplot_outattrs={
                **teeplot_outattrs,
                "what": what,
            },
            teeplot_subdir=teeplot_subdir,
        ) as g:
            for ax in g.axes.flat:
                ax.grid(True, alpha=0.3)
            g.figure.set_size_inches(w=4, h=1.5)
            sns.move_legend(
                g, "center left", bbox_to_anchor=(0.9, 0.5), frameon=False
            )

        with tp.teed(
            sns.relplot,
            data=data,
            x="Step",
            y="Prevalence",
            hue="Class",
            col="Ham. Wt.",
            row=row,
            alpha=0.8,
            dashes=False,
            errorbar=("pi", 100),
            err_kws=dict(alpha=0.1),
            estimator=np.median,
            facet_kws=dict(
                margin_titles=True,
            ),
            kind="line",
            palette=palette,
            teeplot_outattrs={
                **teeplot_outattrs,
                "what": what,
            },
            teeplot_subdir=teeplot_subdir,
        ) as g:
            g.map_dataframe(
                sns.lineplot,
                x="Step",
                y="Prevalence",
                hue="Class",
                style="Seed",
                alpha=0.7,
                dashes=False,
                errorbar=None,
                legend=False,
                linestyle=":",
                linewidth=0.6,
                palette=palette,
            )
            for ax in g.axes.flat:
                ax.grid(True, alpha=0.3)
            if what == "Strain":
                g.set(yscale="log")

            g.set(ylim=(1 / POP_SIZE, 1.1))
            g.figure.suptitle(suptitle)
            if row is not None:
                g.figure.subplots_adjust(hspace=0.16, top=0.9)
                g.figure.set_size_inches(w=5, h=5)
            else:
                g.figure.subplots_adjust(top=0.7)
                g.figure.set_size_inches(w=5, h=2)

            sns.move_legend(
                g,
                "center left",
                bbox_to_anchor=(0.9, 0.5),
                frameon=False,
            )


## Run Simulation and Render Plots across Condition Matrix


In [ ]:
N_REP = 3
N_STEPS = 600
condition_matrix = it.product(
    [5e-5],  # MUTATION_RATE
    [100_000],  # POP_SIZE
    [2],  # N_SITES
)
for MUTATION_RATE, POP_SIZE, N_SITES in tqdm([*condition_matrix]):
    suptitle = (
        f"Pop Size: {POP_SIZE / 1_000_000}M., "
        f"Mutation Rate: {MUTATION_RATE}, "
        f"Num Sites: {N_SITES}"
    )
    display_html(f"<h2>{suptitle}</h2>", raw=True)
    dfs = [
        simulate(
            MUTATION_RATE=MUTATION_RATE,
            N_SITES=N_SITES,
            N_STEPS=N_STEPS,
            POP_SIZE=POP_SIZE,
            seed=rep,
        )
        for rep in range(N_REP)
    ]
    df = pd.concat(dfs)
    render_timeseries_plots(
        df=df,
        suptitle=suptitle,
        teeplot_outattrs={
            "mutation_rate": MUTATION_RATE,
            "n_sites": N_SITES,
            "pop_size": POP_SIZE,
        },
        POP_SIZE=POP_SIZE,
    )
